# 02 — Preprocessing
Clean, encode, scale, and split the dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

data_path = Path('../../data/plant_health_data.csv')
if not data_path.exists():
    data_path = Path('../data/plant_health_data.csv')

df = pd.read_csv(data_path)

In [2]:
# Drop non-predictive columns
df = df.drop(columns=['Timestamp', 'Plant_ID'])
print("Columns after drop:", df.columns.tolist())

Columns after drop: ['Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature', 'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content', 'Electrochemical_Signal', 'Plant_Health_Status']


In [3]:
# Handle missing values
print("Missing before:", df.isnull().sum().sum())
df = df.dropna()
print("Missing after:", df.isnull().sum().sum())

Missing before: 0
Missing after: 0


In [4]:
# Encode target label with logical severity order
label_map = {'Healthy': 0, 'Moderate Stress': 1, 'High Stress': 2}
class_names = np.array(['Healthy', 'Moderate Stress', 'High Stress'])
df['Label'] = df['Plant_Health_Status'].map(label_map)

# Keep a LabelEncoder-compatible object for inverse_transform in later notebooks
le = LabelEncoder()
le.classes_ = class_names

print("Label mapping:", label_map)
df.head()


Label mapping: {'Healthy': 0, 'Moderate Stress': 1, 'High Stress': 2}


,Soil_Moisture,Ambient_Temperature,Soil_Temperature,Humidity,Light_Intensity,Soil_pH,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Chlorophyll_Content,Electrochemical_Signal,Plant_Health_Status,Label
0,27.521109,22.240245,21.900435,55.291904,556.172805,5.581955,10.003650,45.806852,39.076199,35.703006,0.941402,High Stress,2
1,14.835566,21.706763,18.680892,63.949181,596.136721,7.135705,30.712562,25.394393,17.944826,27.993296,0.164899,High Stress,2
2,17.086362,21.180946,15.392939,67.837956,591.124627,5.656852,29.337002,27.573892,35.706530,43.646308,1.081728,High Stress,2
3,15.336156,22.593302,22.778394,58.190811,241.412476,5.584523,16.966621,26.180705,26.257746,37.838095,1.186088,High Stress,2
4,39.822216,28.929001,18.100937,63.772036,444.493830,5.919707,10.944961,37.898907,37.654483,48.265812,1.609805,High Stress,2


In [5]:
# Features and target
X = df.drop(columns=['Plant_Health_Status', 'Label'])
y = df['Label']
print("Features:", X.shape)
print("Target:", y.shape)

Features: (1200, 11)
Target: (1200,)


In [6]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaling done. Mean ~0:", X_scaled.mean(axis=0).round(2))

Scaling done. Mean ~0: [ 0. -0. -0.  0.  0. -0. -0.  0. -0.  0.  0.]


In [7]:
# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (960, 11)
Test size: (240, 11)


In [8]:
import joblib, os
os.makedirs('../../models', exist_ok=True)

# Save processed data and scaler
processed = pd.DataFrame(X_scaled, columns=X.columns)
processed['Label'] = y.values
processed.to_csv('../../data/processed_dataset.csv', index=False)
joblib.dump(scaler, '../../models/scaler.pkl')
joblib.dump((X_train, X_test, y_train, y_test), '../../models/splits.pkl')
joblib.dump(le, '../../models/label_encoder.pkl')
print("Saved: processed_dataset.csv, scaler.pkl, splits.pkl, label_encoder.pkl")

Saved: processed_dataset.csv, scaler.pkl, splits.pkl, label_encoder.pkl
